<a href="https://colab.research.google.com/github/LucaWata/Calculo_Numerico/blob/main/Atividade_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 4 - Animação da Eliminação de Gauss

## Bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import ipywidgets as widgets
from IPython.display import display, Image, clear_output
from google.colab import files

## Configurações da animação

In [ ]:
FRAMES_POR_OPERACAO = 18

FRAMES_INICIAL = 8

FRAMES_FINAL = 25

FPS = 15

CORES = [
    "#4C78A8",
    "#F58518",
    "#54A24B"
]

## Entradas

In [ ]:
eq1 = widgets.Text(
    value="2 1 -1 8",
    description="Eq. 1:",
    layout=widgets.Layout(width="320px")
)

eq2 = widgets.Text(
    value="-3 -1 2 -11",
    description="Eq. 2:",
    layout=widgets.Layout(width="320px")
)

eq3 = widgets.Text(
    value="-2 1 2 -3",
    description="Eq. 3:",
    layout=widgets.Layout(width="320px")
)

botao = widgets.Button(
    description="Gerar GIF",
    button_style="success",
    icon="film",
    layout=widgets.Layout(width="150px")
)

saida = widgets.Output()

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Eliminação de Gauss — sistema 3×3</h3>"
        ),
        widgets.HTML(
            "Digite os coeficientes no formato "
            "<b>a b c d</b>, correspondendo a "
            "<b>ax + by + cz = d</b>."
        ),
        eq1,
        eq2,
        eq3,
        botao,
        saida
    ])
)

## Leitura das equações

In [ ]:
def ler_equacoes():

    linhas = []

    for campo in [eq1, eq2, eq3]:

        valores = list(
            map(float, campo.value.split())
        )

        if len(valores) != 4:
            raise ValueError(
                "Cada equação deve possuir exatamente "
                "4 números: a b c d"
            )

        linhas.append(valores)

    return np.array(
        linhas,
        dtype=float
    )

## Etapas de Gauss

In [ ]:
def gerar_etapas(A):

    A = A.astype(float).copy()

    etapas = []

    # Estado inicial
    etapas.append({
        "antes": A.copy(),
        "depois": A.copy(),
        "operacao": "Matriz inicial",
        "pivo": None,
        "linha_alterada": None
    })

    n = 3

    for coluna in range(n):

        # ESCOLHA DO PIVÔ
        linha_pivo = coluna + np.argmax(
            np.abs(A[coluna:, coluna])
        )

        if abs(A[linha_pivo, coluna]) < 1e-10:
            continue

        # TROCA DE LINHAS
        if linha_pivo != coluna:

            antes = A.copy()

            A[[coluna, linha_pivo]] = \
                A[[linha_pivo, coluna]]

            depois = A.copy()

            etapas.append({
                "antes": antes,
                "depois": depois,
                "operacao":
                    f"L{coluna+1} ↔ L{linha_pivo+1}",
                "pivo": (
                    coluna,
                    coluna
                ),
                "linha_alterada": None
            })

        # ELIMINAÇÃO

        for linha in range(
            coluna + 1,
            n
        ):

            if abs(A[linha, coluna]) < 1e-10:
                continue

            fator = (
                A[linha, coluna]
                / A[coluna, coluna]
            )

            antes = A.copy()

            A[linha] = (
                A[linha]
                - fator * A[coluna]
            )

            A[
                np.abs(A) < 1e-10
            ] = 0

            depois = A.copy()

            etapas.append({
                "antes": antes,
                "depois": depois,
                "operacao":
                    formatar_operacao(
                        linha + 1,
                        coluna + 1,
                        fator
                    ),
                "pivo": (
                    coluna,
                    coluna
                ),
                "linha_alterada": linha
            })

    return etapas

## Interpolação

In [ ]:
def suavizar(t):

    # Smoothstep:
    # começa devagar, acelera no meio
    # e termina devagar.
    return (
        3 * t**2
        - 2 * t**3
    )


def criar_frames(etapas):

    frames = []

    # MATRIZ INICIAL
    matriz_inicial = etapas[0]["antes"]

    for _ in range(FRAMES_INICIAL):

        frames.append({
            "matriz": matriz_inicial.copy(),
            "operacao": "Matriz inicial",
            "pivo": None,
            "linha_alterada": None
        })

    # INTERPOLA CADA OPERAÇÃO
    for etapa in etapas[1:]:

        antes = etapa["antes"]
        depois = etapa["depois"]

        for k in range(
            1,
            FRAMES_POR_OPERACAO + 1
        ):

            t = k / FRAMES_POR_OPERACAO

            # Interpolação suave
            s = suavizar(t)

            matriz = (
                (1 - s) * antes
                + s * depois
            )

            frames.append({
                "matriz": matriz,
                "operacao": etapa["operacao"],
                "pivo": etapa["pivo"],
                "linha_alterada":
                    etapa["linha_alterada"]
            })

    # ÚLTIMA MATRIZ
    ultima = etapas[-1]

    for _ in range(FRAMES_FINAL):

        frames.append({
            "matriz": ultima["depois"].copy(),
            "operacao":
                "Forma triangular superior",
            "pivo": ultima["pivo"],
            "linha_alterada": None
        })

    return frames

## Resolver o sistema

In [ ]:
def resolver(A):

    M = A[:, :3]
    b = A[:, 3]

    posto = np.linalg.matrix_rank(M)
    posto_aum = np.linalg.matrix_rank(A)

    if posto < posto_aum:
        return "Sem solução", None

    if posto < 3:
        return "Infinitas soluções", None

    return (
        "Solução única",
        np.linalg.solve(M, b)
    )

## Limite dos gráficos

In [ ]:
def calcular_limite(A):

    try:

        status, solucao = resolver(A)

        if solucao is not None:

            maior = np.max(
                np.abs(solucao)
            )

            if maior > 3:

                return min(
                    15,
                    max(6, maior + 3)
                )

    except:
        pass

    maior = np.max(
        np.abs(A[:, 3])
    )

    return min(
        12,
        max(6, maior + 3)
    )

## Desenha um plano

In [ ]:
def desenhar_plano(
    ax,
    coeficientes,
    indice,
    limite
):

    a, b, c, d = coeficientes

    coef = np.array([
        abs(a),
        abs(b),
        abs(c)
    ])

    if np.max(coef) < 1e-10:
        return

    dominante = np.argmax(coef)

    # Mais pontos = plano mais suave
    u = np.linspace(
        -limite,
        limite,
        45
    )

    v = np.linspace(
        -limite,
        limite,
        45
    )

    U, V = np.meshgrid(
        u,
        v
    )

    if dominante == 0:

        Y = U
        Z = V

        X = (
            d
            - b * Y
            - c * Z
        ) / a

    elif dominante == 1:

        X = U
        Z = V

        Y = (
            d
            - a * X
            - c * Z
        ) / b

    else:

        X = U
        Y = V

        Z = (
            d
            - a * X
            - b * Y
        ) / c

    # Evita planos explodindo visualmente
    X = np.clip(
        X,
        -limite,
        limite
    )

    Y = np.clip(
        Y,
        -limite,
        limite
    )

    Z = np.clip(
        Z,
        -limite,
        limite
    )

    cor = CORES[indice]

    # SUPERFÍCIE
    ax.plot_surface(
        X,
        Y,
        Z,
        color=cor,
        alpha=0.28,
        linewidth=0,
        antialiased=True,
        shade=True
    )

    # CONTORNO
    for k in [0, -1]:

        ax.plot(
            X[k, :],
            Y[k, :],
            Z[k, :],
            color=cor,
            alpha=0.55,
            linewidth=1
        )

        ax.plot(
            X[:, k],
            Y[:, k],
            Z[:, k],
            color=cor,
            alpha=0.55,
            linewidth=1
        )

## Gerar GIF

In [ ]:
def gerar_gif(_):

    with saida:

        clear_output(
            wait=True
        )

        # LÊ O SISTEMA
        try:

            A_original = ler_equacoes()

        except Exception as erro:

            print(
                "Erro:",
                erro
            )

            return

        # ETAPAS E FRAMES
        etapas = gerar_etapas(
            A_original
        )

        frames = criar_frames(
            etapas
        )

        status, solucao = resolver(
            A_original
        )

        limite = calcular_limite(
            A_original
        )

        total_frames = len(frames)

        # FIGURA
        fig = plt.figure(
            figsize=(15, 7.5),
            facecolor="white"
        )

        gs = fig.add_gridspec(
            1,
            2,
            width_ratios=[
                0.85,
                1.35
            ],
            wspace=0.04
        )

        ax_matriz = fig.add_subplot(
            gs[0]
        )

        ax_3d = fig.add_subplot(
            gs[1],
            projection="3d"
        )

        # ATUALIZA CADA FRAME
        def atualizar(frame_num):

            ax_matriz.clear()
            ax_3d.clear()

            frame = frames[
                frame_num
            ]

            M = frame["matriz"]

            # MATRIZ
            ax_matriz.axis(
                "off"
            )

            tabela = ax_matriz.table(
                cellText=[
                    [
                        fmt(M[i, 0]),
                        fmt(M[i, 1]),
                        fmt(M[i, 2]),
                        fmt(M[i, 3])
                    ]
                    for i in range(3)
                ],
                colLabels=[
                    "x",
                    "y",
                    "z",
                    "b"
                ],
                loc="center",
                cellLoc="center"
            )

            tabela.auto_set_font_size(
                False
            )

            tabela.set_fontsize(
                15
            )

            tabela.scale(
                1.45,
                2.15
            )

            # CABEÇALHO
            for j in range(4):

                tabela[
                    (0, j)
                ].set_text_props(
                    weight="bold",
                    fontsize=14
                )

                tabela[
                    (0, j)
                ].set_facecolor(
                    "#E9EEF5"
                )

            # PIVÔ
            if frame["pivo"] is not None:

                lp, cp = frame[
                    "pivo"
                ]

                celula = tabela[
                    (lp + 1, cp)
                ]

                celula.set_text_props(
                    weight="bold",
                    fontsize=18
                )

                celula.set_linewidth(
                    3
                )

                celula.set_edgecolor(
                    "#D62728"
                )

            # LINHA ALTERADA
            if (
                frame["linha_alterada"]
                is not None
            ):

                la = frame[
                    "linha_alterada"
                ]

                for j in range(4):

                    tabela[
                        (la + 1, j)
                    ].set_linewidth(
                        2
                    )

                    tabela[
                        (la + 1, j)
                    ].set_edgecolor(
                        "#555555"
                    )

            # TÍTULO
            ax_matriz.set_title(
                "Matriz aumentada",
                fontsize=18,
                fontweight="bold",
                pad=18
            )

            # OPERAÇÃO
            ax_matriz.text(
                0.5,
                0.16,
                frame["operacao"],
                transform=ax_matriz.transAxes,
                ha="center",
                va="center",
                fontsize=14,
                fontweight="bold"
            )

            ax_matriz.text(
                0.5,
                0.08,
                f"Frame {frame_num + 1} / "
                f"{total_frames}",
                transform=ax_matriz.transAxes,
                ha="center",
                fontsize=10
            )

            # PLANOS
            for i in range(3):

                desenhar_plano(
                    ax_3d,
                    M[i],
                    i,
                    limite
                )

            # SOLUÇÃO
            if (
                status == "Solução única"
                and solucao is not None
            ):

                # Halo
                ax_3d.scatter(
                    solucao[0],
                    solucao[1],
                    solucao[2],
                    s=300,
                    color="white",
                    alpha=0.8,
                    edgecolor="none"
                )

                # Ponto
                ax_3d.scatter(
                    solucao[0],
                    solucao[1],
                    solucao[2],
                    s=100,
                    color="#222222",
                    edgecolor="white",
                    linewidth=1.5,
                    marker="o"
                )

                # Linha vertical
                ax_3d.plot(
                    [
                        solucao[0],
                        solucao[0]
                    ],
                    [
                        solucao[1],
                        solucao[1]
                    ],
                    [
                        -limite,
                        solucao[2]
                    ],
                    linestyle="--",
                    linewidth=1,
                    color="#444444",
                    alpha=0.45
                )

            # EIXOS
            ax_3d.set_xlim(
                -limite,
                limite
            )

            ax_3d.set_ylim(
                -limite,
                limite
            )

            ax_3d.set_zlim(
                -limite,
                limite
            )

            ax_3d.set_xlabel(
                "x",
                fontsize=11,
                labelpad=6
            )

            ax_3d.set_ylabel(
                "y",
                fontsize=11,
                labelpad=6
            )

            ax_3d.set_zlabel(
                "z",
                fontsize=11,
                labelpad=6
            )

            ax_3d.set_title(
                "Planos do sistema",
                fontsize=18,
                fontweight="bold",
                pad=15
            )

            # CÂMERA
            ax_3d.view_init(
                elev=23,
                azim=-55
            )

            try:

                ax_3d.set_box_aspect(
                    (1, 1, 1)
                )

            except:
                pass

            ax_3d.grid(
                True,
                alpha=0.18
            )

            # LEGENDA
            for i in range(3):

                ax_3d.plot(
                    [],
                    [],
                    [],
                    color=CORES[i],
                    linewidth=5,
                    label=f"Equação {i+1}"
                )

            if (
                status ==
                "Solução única"
            ):

                ax_3d.scatter(
                    [],
                    [],
                    [],
                    color="#222222",
                    s=70,
                    label="Solução"
                )

            ax_3d.legend(
                loc="upper left",
                fontsize=9,
                framealpha=0.9
            )

            # TEXTO FINAL
            if (
                frame_num ==
                total_frames - 1
            ):

                if (
                    status ==
                    "Solução única"
                    and solucao is not None
                ):

                    texto = (
                        "Solução:  "
                        f"x = {solucao[0]:.3f}   "
                        f"y = {solucao[1]:.3f}   "
                        f"z = {solucao[2]:.3f}"
                    )

                else:

                    texto = status

                fig.text(
                    0.68,
                    0.015,
                    texto,
                    ha="center",
                    fontsize=13,
                    fontweight="bold"
                )

            # TÍTULO GERAL
            fig.suptitle(
                "Eliminação de Gauss",
                fontsize=21,
                fontweight="bold",
                y=0.98
            )

            fig.tight_layout(
                rect=[
                    0,
                    0.03,
                    1,
                    0.94
                ]
            )

        # ANIMAÇÃO
        animacao = FuncAnimation(
            fig,
            atualizar,
            frames=total_frames,
            interval=1000 / FPS,
            repeat=False
        )

        # SALVA O GIF
        caminho = (
            "eliminacao_gauss_fluido.gif"
        )

        animacao.save(
            caminho,
            writer=PillowWriter(
                fps=FPS
            )
        )

        plt.close(fig)

        # MOSTRA RESULTADO
        print(
            "GIF gerado com sucesso!"
        )

        print()

        print(
            f"Total de frames: "
            f"{total_frames}"
        )

        print(
            f"FPS: {FPS}"
        )

        print()

        print(
            "Sistema utilizado:"
        )

        for i, linha in enumerate(
            A_original
        ):

            print(
                f"Eq. {i+1}: "
                f"{fmt(linha[0])}x + "
                f"{fmt(linha[1])}y + "
                f"{fmt(linha[2])}z = "
                f"{fmt(linha[3])}"
            )

        print()

        print(
            "Resultado:",
            status
        )

        if solucao is not None:

            print(
                f"x = {solucao[0]:.6f}   "
                f"y = {solucao[1]:.6f}   "
                f"z = {solucao[2]:.6f}"
            )

        display(
            Image(
                filename=caminho
            )
        )

        # DOWNLOAD
        botao_download = widgets.Button(
            description="Baixar GIF",
            button_style="info",
            icon="download"
        )

        def baixar(_):

            files.download(
                caminho
            )

        botao_download.on_click(
            baixar
        )

        display(
            botao_download
        )

## Botao de download

In [31]:
botao.on_click(
    gerar_gif
)